%md
# Notebook 15: Scheduled Pipeline with Quality Gates

**Question:** Can the Notebook 13 flow run on its own, on a schedule, and stop itself when the data is wrong instead of quietly publishing bad results?

**Pipeline:** CMS download → bronze → silver → gold (CPAP peer scores) → shareable view

**Principles:**
- **Idempotent:** running it twice gives the same result as running it once.
- **Fail loudly:** every quality gate stops the pipeline on failure.
- **Keep history:** every run is logged, never overwritten.
- **Publishing rule:** no provider identifiers in any output.

In [0]:
from pyspark.sql import functions as F
import os, urllib.request, uuid, datetime

SCHEMA = "workspace.default"
URL = "https://data.cms.gov/sites/default/files/2026-07/1ae6eac7-c8a5-4f1c-a82e-617f4bb65a5f/mup_dme_ry26_p05_v10_dy24_rfrhpr.csv"
DEST = "/Volumes/workspace/default/raw_data/dme_referring_provider_service_2024.csv"
BRONZE = f"{SCHEMA}.dme_referring_2024_bronze"
RUN_LOG = f"{SCHEMA}.pipeline_run_log"
RUN_ID = str(uuid.uuid4())[:8]

spark.sql(f"""CREATE TABLE IF NOT EXISTS {RUN_LOG}
              (run_id STRING, run_at TIMESTAMP, gate STRING, status STRING, detail STRING)""")

def gate(name, passed, detail):
    status = "PASS" if passed else "FAIL"
    spark.createDataFrame([(RUN_ID, datetime.datetime.now(datetime.timezone.utc), name, status, detail)],
                          "run_id string, run_at timestamp, gate string, status string, detail string") \
         .write.mode("append").saveAsTable(RUN_LOG)
    print(f"[{status}] {name}: {detail}")
    if not passed:
        raise Exception(f"Quality gate failed: {name} - {detail}")

print(f"Run {RUN_ID} started")

In [0]:
if not os.path.exists(DEST):
    urllib.request.urlretrieve(URL, DEST)
size_mb = os.path.getsize(DEST) / 1e6
gate("1a_file_present", size_mb > 100, f"{size_mb:,.0f} MB")

raw = (spark.read.option("header", True).option("inferSchema", True)
       .option("quote", '"').option("escape", '"').csv(DEST))
n = raw.count()
gate("1b_row_count", 1_000_000 <= n <= 2_000_000, f"{n:,} rows (expected 1.0M-2.0M)")

bad = raw.filter(~F.col("Tot_Suplrs").cast("string").rlike("^[0-9]+$")).count()
gate("1c_parsing_clean", bad == 0, f"{bad:,} shifted rows")

if spark.catalog.tableExists(BRONZE) and spark.table(BRONZE).count() == n:
    print("Bronze already current: rewrite skipped")
else:
    raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(BRONZE)
    print("Bronze written")

In [0]:
display(spark.table(RUN_LOG).orderBy(F.desc("run_at")))

In [0]:
SILVER = f"{SCHEMA}.dme_referring_2024_silver"
CPAP_CODES = ["E0601", "E0562", "A4604", "A7030", "A7031", "A7032", "A7033", "A7034",
              "A7035", "A7036", "A7037", "A7038", "A7039", "A7046"]

silver = (spark.table(BRONZE)
    .withColumn("Rfrg_NPI", F.col("Rfrg_NPI").cast("string"))
    .select("Rfrg_NPI", "Rfrg_Prvdr_State_Abrvtn", "Rfrg_Prvdr_RUCA_Cat", "Rfrg_Prvdr_Spclty_Desc",
            "RBCS_Desc", "HCPCS_CD", "HCPCS_Desc", "Suplr_Rentl_Ind",
            "Tot_Suplrs", "Tot_Suplr_Benes", "Tot_Suplr_Clms", "Tot_Suplr_Srvcs",
            "Avg_Suplr_Mdcr_Stdzd_Amt"))
silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER)
silver = spark.table(SILVER)

total = silver.count()
hidden = silver.filter(F.col("Tot_Suplr_Benes").isNull()).count()
share = hidden / total
gate("2a_suppression_share", 0.59 <= share <= 0.79,
     f"{share:.1%} of rows with hidden patient counts (expected 59-79%)")

cov = (silver.filter(F.col("HCPCS_CD").isin(CPAP_CODES))
       .agg(F.sum("Tot_Suplr_Srvcs").alias("all_srv"),
            F.sum(F.when(F.col("Tot_Suplr_Benes").isNotNull(), F.col("Tot_Suplr_Srvcs")).otherwise(0)).alias("visible_srv"))
       .collect()[0])
coverage = cov["visible_srv"] / cov["all_srv"]
gate("2b_cpap_coverage", coverage >= 0.75, f"{coverage:.1%} of CPAP services visible (minimum 75%)")

In [0]:
from pyspark.sql import Window

GOLD = f"{SCHEMA}.dme_cpap_peer_scores_gold"
VIEW = f"{SCHEMA}.cpap_resupply_share_v"
ANNUAL_MAX = {"A7030": 4, "A7031": 12, "A7032": 24, "A7033": 24, "A7034": 4, "A7035": 2,
              "A7036": 2, "A7037": 4, "A7038": 24, "A7039": 2, "A7046": 2, "A4604": 4}
limits = spark.createDataFrame(list(ANNUAL_MAX.items()), ["HCPCS_CD", "annual_max_per_patient"])

supply = (silver.filter(F.col("HCPCS_CD").isin(CPAP_CODES) & F.col("Tot_Suplr_Benes").isNotNull())
          .join(limits, "HCPCS_CD")
          .withColumn("srv_per_patient", F.col("Tot_Suplr_Srvcs") / F.col("Tot_Suplr_Benes"))
          .withColumn("pct_of_max", F.col("srv_per_patient") / F.col("annual_max_per_patient")))

peer = Window.partitionBy("HCPCS_CD", "Rfrg_Prvdr_Spclty_Desc")
scored = (supply
    .withColumn("peer_n", F.count("*").over(peer))
    .withColumn("peer_median", F.percentile_approx("srv_per_patient", 0.5).over(peer))
    .withColumn("abs_dev", F.abs(F.col("srv_per_patient") - F.col("peer_median")))
    .withColumn("peer_mad", F.percentile_approx("abs_dev", 0.5).over(peer))
    .filter(F.col("peer_n") >= 30)
    .withColumn("robust_z", F.when(F.col("peer_mad") > 0,
                0.6745 * (F.col("srv_per_patient") - F.col("peer_median")) / F.col("peer_mad"))))
scored.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD)
print(f"Gold rebuilt: {spark.table(GOLD).count():,} scored rows")

# Gate 3: privacy
v = spark.table(VIEW)
id_cols = [c for c in v.columns if "NPI" in c.upper() or "NAME" in c.upper()]
gate("3a_no_identifiers", len(id_cols) == 0, f"identifier-like columns in shared view: {id_cols or 'none'}")

min_group = v.agg(F.min("prescribers")).collect()[0][0]
gate("3b_min_group_size", min_group >= 11, f"smallest shared group: {min_group} prescribers")

names_visible = spark.sql(f"SELECT COUNT(DISTINCT Rfrg_Prvdr_Last_Name_Org) FROM {BRONZE}").collect()[0][0]
gate("3c_masks_intact", names_visible == 1, f"distinct names visible to pipeline user: {names_visible}")

In [0]:
gate("9_run_complete", True, f"All gates passed | gold rows: {spark.table(GOLD).count():,}")

display(spark.table(RUN_LOG).filter(F.col("run_id") == RUN_ID).orderBy("run_at"))